# Grilla de min_data_in_leaf sobre el dataset con Lags/Deltas

Con las ~751 variables (vs ~151 originales), los hiperparámetros de A y B quedaron desregulados: la ganancia media subió (452,65M → 458,92M) pero el desvío se disparó (2,28M → 8,29M) y el % de Gain de canaritos también subió bastante (A: 40,14%→50,72%, B: 32,46%→43,43%).

La hipótesis: `min_data_in_leaf` está muy bajo para la cantidad de columnas actual. Esta notebook prueba valores más altos, para cada modelo por separado, midiendo en cada uno:
1. **% de Gain de canaritos** (una semilla, diagnóstico rápido)
2. **Ganancia media y desvío a 10 semillas** (validación completa)

Igual que en las grillas anteriores, guarda resultados incrementalmente (`fwrite(append=...)`) para poder cortar y retomar si hace falta.

- **Modelo A**: `min_data_in_leaf` actual = 1000 → probamos `c(1000, 2000, 4000, 6000, 9000)`
- **Modelo B**: `min_data_in_leaf` actual = 460 → probamos `c(460, 1000, 2000, 3500, 5000, 7500)` (B es el que más se descontroló, así que el rango es más amplio)

### Setup del ambiente y carga de datos

In [1]:
rm(list = ls())
gc()

PARAM_carpeta_datasets <- "C:/Users/Laura/Documents/DMEyF/datasets/"
PARAM_carpeta_exp      <- "C:/Users/Laura/Documents/DMEyF/exp/"

dir.create(PARAM_carpeta_datasets, showWarnings = FALSE, recursive = TRUE)
dir.create(PARAM_carpeta_exp, showWarnings = FALSE, recursive = TRUE)

require("data.table")
require("lightgbm")

dataset <- fread(paste0(PARAM_carpeta_datasets, "competencia_01_con_lags.csv.gz"), stringsAsFactors = TRUE)

if (!("clase_ternaria" %in% colnames(dataset))) {
  stop("No se encontro clase_ternaria -- revisar competencia_01_con_lags.csv.gz")
}

dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]

campos_buenos <- setdiff(
  colnames(dataset),
  c("clase_ternaria", "clase01", "numero_de_cliente")
)
campos_buenos <- setdiff(campos_buenos, c("cprestamos_personales", "mprestamos_personales"))

cat("campos_buenos:", length(campos_buenos), "variables\n")

dataset_train <- dataset[foto_mes == 202104]
dfuture       <- dataset[foto_mes == 202106]

dtrain <- lgb.Dataset(
  data = data.matrix(dataset_train[, campos_buenos, with = FALSE]),
  label = dataset_train[, clase01]
)

cat("Filas de entrenamiento:", nrow(dataset_train), "\n")

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,653321,34.9,1447531,77.4,1118889,59.8
Vcells,1199649,9.2,8388608,64.0,1928501,14.8


Caricamento del pacchetto richiesto: data.table

Caricamento del pacchetto richiesto: lightgbm



campos_buenos: 751 variables
Filas de entrenamiento: 163284 


### Hiperparámetros base y funciones auxiliares

In [2]:
param_comun <- list(
  boosting= "gbdt", objective= "binary", metric= "auc",
  first_metric_only= FALSE, boost_from_average= TRUE, feature_pre_filter= FALSE,
  force_row_wise= TRUE, verbosity= -100,
  max_depth= -1L, max_bin= 31L,
  min_sum_hessian_in_leaf = 0.001,
  pos_bagging_fraction= 1.0, neg_bagging_fraction= 1.0, is_unbalance= FALSE,
  drop_rate= 0.1, max_drop= 50, skip_drop= 0.5, extra_trees= FALSE
)

param_A_base <- modifyList(param_comun, list(
  num_iterations = 1351, learning_rate = 0.02, feature_fraction = 0.5153582,
  num_leaves = 1601, min_data_in_leaf = 1000, scale_pos_weight = 0.5,
  min_gain_to_split = 0.2654667, lambda_l1 = 3.26227, lambda_l2 = 7.540213,
  bagging_fraction = 0.8329823, bagging_freq = 5
))

param_B_base <- modifyList(param_comun, list(
  num_iterations = 600, learning_rate = 0.02, feature_fraction = 0.1541961,
  num_leaves = 1684, min_data_in_leaf = 460, scale_pos_weight = 0.5,
  min_gain_to_split = 0.2158453, lambda_l1 = 1.520327, lambda_l2 = 7.548036,
  bagging_fraction = 0.8425105, bagging_freq = 6
))

CORTE_FIJO <- 12000
SEMILLAS_10 <- c(585581, 249449, 655649, 503039, 798191,
                  112909, 334667, 421303, 667021, 887923)
SEMILLA_CANARITOS <- 585581

calcular_ganancia <- function(tb, columna_orden, corte) {
  tb2 <- copy(tb)
  setorderv(tb2, columna_orden, order = -1)
  tb2[, Predicted := 0L]
  tb2[1:corte, Predicted := 1L]
  tb2[Predicted == 1L, sum(ifelse(clase_ternaria == "BAJA+2", 1072500, -27500))]
}

diagnostico_canaritos_pct <- function(dataset_train, campos_buenos, param, seed) {
  dt <- copy(dataset_train)
  set.seed(seed, "L'Ecuyer-CMRG")
  for (campo in campos_buenos) {
    dt[, paste0("canarito_", campo) := runif(.N)]
  }
  campos_canaritos <- paste0("canarito_", campos_buenos)
  campos_totales <- c(campos_buenos, campos_canaritos)

  dtrain_diag <- lgb.Dataset(
    data = data.matrix(dt[, campos_totales, with = FALSE]),
    label = dt[, clase01]
  )

  p <- param; p$seed <- seed
  set.seed(seed, "L'Ecuyer-CMRG")
  modelo_diag <- lgb.train(data = dtrain_diag, param = p)

  tb_imp <- lgb.importance(modelo_diag)
  tb_imp[, es_canarito := grepl("^canarito_", Feature)]
  gain_total <- sum(tb_imp$Gain)
  gain_canaritos <- sum(tb_imp[es_canarito == TRUE, Gain])

  rm(modelo_diag, dtrain_diag, dt)
  gc(full = TRUE, verbose = FALSE)

  gain_canaritos / gain_total
}

evaluar_min_data <- function(param_base, valores_min_data, tag, archivo) {

  ya_hechas <- integer(0)
  if (file.exists(archivo)) {
    tb_prev <- fread(archivo)
    ya_hechas <- tb_prev$min_data_in_leaf
  }

  for (mdl in valores_min_data) {

    if (mdl %in% ya_hechas) {
      cat(tag, "min_data_in_leaf =", mdl, "-- ya calculado, salteando\n")
      next
    }

    p <- modifyList(param_base, list(min_data_in_leaf = mdl))

    pct_canaritos <- diagnostico_canaritos_pct(dataset_train, campos_buenos, p, SEMILLA_CANARITOS)

    ganancias <- numeric(0)
    for (SEM in SEMILLAS_10) {
      p_sem <- p; p_sem$seed <- SEM
      set.seed(SEM, "L'Ecuyer-CMRG")
      modelo <- lgb.train(data = dtrain, param = p_sem)
      pred <- predict(modelo, data.matrix(dfuture[, campos_buenos, with = FALSE]))
      tb <- dfuture[, list(clase_ternaria)]
      tb[, prob := pred]
      gan <- calcular_ganancia(tb, "prob", CORTE_FIJO)
      ganancias <- c(ganancias, gan)
      rm(modelo)
      gc(full = TRUE, verbose = FALSE)
    }

    fila <- data.table(
      min_data_in_leaf = mdl,
      pct_canaritos = pct_canaritos,
      ganancia_media = mean(ganancias),
      ganancia_desvio = sd(ganancias)
    )

    fwrite(fila, file = archivo, sep = "\t", append = file.exists(archivo))

    cat(tag, "min_data_in_leaf =", mdl,
        " pct_canaritos =", round(pct_canaritos*100, 2), "%",
        " ganancia_media =", mean(ganancias),
        " ganancia_desvio =", sd(ganancias), "\n")
  }
}

### Grilla Modelo A

In [3]:
evaluar_min_data(
  param_A_base,
  c(1000, 2000, 4000, 6000, 9000),
  "Modelo A",
  paste0(PARAM_carpeta_exp, "grilla_lags_ModeloA.txt")
)

Modelo A min_data_in_leaf = 1000  pct_canaritos = 50.72 %  ganancia_media = 448470000  ganancia_desvio = 6003712 
Modelo A min_data_in_leaf = 2000  pct_canaritos = 47.64 %  ganancia_media = 446600000  ganancia_desvio = 3360556 
Modelo A min_data_in_leaf = 4000  pct_canaritos = 40.77 %  ganancia_media = 444400000  ganancia_desvio = 4490731 
Modelo A min_data_in_leaf = 6000  pct_canaritos = 34.34 %  ganancia_media = 444400000  ganancia_desvio = 2432192 
Modelo A min_data_in_leaf = 9000  pct_canaritos = 27.46 %  ganancia_media = 443630000  ganancia_desvio = 3156316 


### Grilla Modelo B

In [4]:
evaluar_min_data(
  param_B_base,
  c(460, 1000, 2000, 3500, 5000, 7500),
  "Modelo B",
  paste0(PARAM_carpeta_exp, "grilla_lags_ModeloB.txt")
)

Modelo B min_data_in_leaf = 460  pct_canaritos = 43.43 %  ganancia_media = 459250000  ganancia_desvio = 5803304 
Modelo B min_data_in_leaf = 1000  pct_canaritos = 37.95 %  ganancia_media = 464090000  ganancia_desvio = 4097818 
Modelo B min_data_in_leaf = 2000  pct_canaritos = 30.05 %  ganancia_media = 464090000  ganancia_desvio = 5424932 
Modelo B min_data_in_leaf = 3500  pct_canaritos = 22.45 %  ganancia_media = 461560000  ganancia_desvio = 3246605 
Modelo B min_data_in_leaf = 5000  pct_canaritos = 17.2 %  ganancia_media = 459470000  ganancia_desvio = 5913459 
Modelo B min_data_in_leaf = 7500  pct_canaritos = 11.65 %  ganancia_media = 455510000  ganancia_desvio = 2286895 


### Resultados

In [5]:
cat("--- Modelo A ---\n")
print(fread(paste0(PARAM_carpeta_exp, "grilla_lags_ModeloA.txt")))

cat("\n--- Modelo B ---\n")
print(fread(paste0(PARAM_carpeta_exp, "grilla_lags_ModeloB.txt")))

--- Modelo A ---
   min_data_in_leaf pct_canaritos ganancia_media ganancia_desvio
              <int>         <num>          <int>           <num>
1:             1000     0.5072161      448470000         6003712
2:             2000     0.4764294      446600000         3360556
3:             4000     0.4076948      444400000         4490731
4:             6000     0.3433835      444400000         2432192
5:             9000     0.2746183      443630000         3156316

--- Modelo B ---
   min_data_in_leaf pct_canaritos ganancia_media ganancia_desvio
              <int>         <num>          <int>           <num>
1:              460     0.4343356      459250000         5803304
2:             1000     0.3795178      464090000         4097818
3:             2000     0.3005172      464090000         5424932
4:             3500     0.2244597      461560000         3246605
5:             5000     0.1719642      459470000         5913459
6:             7500     0.1165252      455510000       

### Punto de comparación

| | min_data_in_leaf actual | Ganancia media (con lags) | Desvío (con lags) |
|---|---|---|---|
| Modelo A | 1000 | 448.470.000 | 6.003.712 |
| Modelo B | 460 | 459.250.000 | 5.803.304 |
| **Ensamble actual** | | **458.920.000** | **8.293.478** |

Buscamos, para cada modelo, el `min_data_in_leaf` que reduzca el desvío (y ojalá también el % de canaritos) sin perder demasiada ganancia media. Con esos valores nuevos armamos el ensamble final y lo comparamos contra:
- 452.650.000 / 2.275.107 (mejor resultado sin lags/deltas)
- 458.920.000 / 8.293.478 (con lags/deltas, sin retunear)

El objetivo es quedarnos con lo mejor de los dos mundos: la ganancia extra que aportan las variables nuevas, con un desvío controlado.